# Занятие 9. Группировка, агрегаты и мини-сводные таблицы

## Краткая теория по ноутбуку

На этом занятии мы переходим от просмотра отдельных строк к **аналитике по группам**.  
Нас интересуют уже не отдельные заказы, а ответы на вопросы:

- сколько заказов у каждого менеджера;
- какая общая выручка по категориям;
- кто лидер по выручке;
- как сравнить данные по двум признакам сразу.

## Что важно понять

- **Агрегатные функции** помогают сворачивать много строк в один итог: сумма, среднее, максимум, минимум, количество.
- **Группировка** (`groupby`) объединяет строки с одинаковым значением в выбранном столбце.
- **Агрегация после группировки** позволяет быстро получить бизнес-итоги по менеджерам, месяцам и категориям.
- **Мини-сводная таблица** — это удобная форма представления результатов, когда по строкам идёт одна группа, а по столбцам — другая.
- В системах с ИИ такие операции нужны для анализа датасета, построения признаков и проверки качества данных до обучения моделей.


## Практическая ячейка 1. Загружаем Excel и считаем базовые агрегаты

### Что делаем
Открываем Excel-файл занятия, считываем лист `sales_data`, создаём столбец `revenue` и считаем:

- количество заказов;
- общую выручку;
- среднюю выручку заказа;
- максимальную и минимальную выручку.

### Функции, методы и синтаксис
- `import pandas as pd` — подключаем библиотеку `pandas`.
- `Path("имя_файла.xlsx")` — создаём путь к файлу.
- `pd.read_excel(file_path, sheet_name="sales_data", header=1)` — читаем лист Excel в `DataFrame`.
- `df["revenue"] = df["quantity"] * df["price"]` — создаём вычисляемый столбец.
- `sum()`, `mean()`, `max()`, `min()`, `count()` — базовые агрегатные функции.


In [ ]:
from pathlib import Path

import pandas as pd

file_path = Path("../../xls/lesson_09_groupby_pivot.xlsx")
sheet_name = "sales_data"
refresh_data = False

In [ ]:
import random
import sys
from datetime import datetime, timedelta

import numpy as np

if not refresh_data:
    sys.exit("Пропуск, нет необходимости обновлять данные")
    
random.seed(42)
np.random.seed(42)

# ---------- Настройки для генерации ----------
# Период: весь 2025 год
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 12, 31)
date_list = [
    start_date + timedelta(days=i) for i in range((end_date - start_date).days + 1)
]

# Имена менеджеров (15)
names = [
    "Анна",
    "Борис",
    "Светлана",
    "Дмитрий",
    "Елена",
    "Игорь",
    "Ольга",
    "Павел",
    "Мария",
    "Сергей",
    "Татьяна",
    "Алексей",
    "Наталья",
    "Владимир",
    "Ксения",
]

# Города и регионы (15 городов, 8 регионов)
cities_regions = {
    "Москва": "Центральный",
    "Санкт-Петербург": "Северо-Западный",
    "Казань": "Приволжский",
    "Новосибирск": "Сибирский",
    "Екатеринбург": "Уральский",
    "Краснодар": "Южный",
    "Нижний Новгород": "Приволжский",
    "Челябинск": "Уральский",
    "Самара": "Приволжский",
    "Омск": "Сибирский",
    "Ростов-на-Дону": "Южный",
    "Уфа": "Приволжский",
    "Красноярск": "Сибирский",
    "Пермь": "Приволжский",
    "Воронеж": "Центральный",
}
cities = list(cities_regions.keys())

# Расширенные категории и товары (6 категорий, по 5-6 товаров)
products_dict = {
    "Ноутбуки": [
        "Notebook Air",
        "Notebook Pro",
        "Notebook Lite",
        "Notebook Ultra",
        "Notebook Work",
    ],
    "Мониторы": [
        "Monitor 27",
        "Monitor 24",
        "Monitor 32",
        "Monitor 21",
        "Monitor Curved",
    ],
    "Периферия": ["Mouse Pro", "Keyboard", "Dock Station", "Webcam", "USB Hub"],
    "Принтеры": ["Printer Laser", "Printer Ink", "Printer MFP", "Printer Photo"],
    "Аксессуары": [
        "Power Bank",
        "Bagpack",
        "Screen Protector",
        "Cooling Pad",
        "Cable Set",
    ],
    "Серверы": [
        "Server Rack",
        "Server Blade",
        "Storage NAS",
        "Network Switch",
        "Firewall",
    ],
}

# Базовая цена для каждого товара
base_prices = {
    "Notebook Air": 60000,
    "Notebook Pro": 95000,
    "Notebook Lite": 45000,
    "Notebook Ultra": 120000,
    "Notebook Work": 75000,
    "Monitor 27": 18000,
    "Monitor 24": 16000,
    "Monitor 32": 28000,
    "Monitor 21": 12000,
    "Monitor Curved": 22000,
    "Mouse Pro": 2500,
    "Keyboard": 3500,
    "Dock Station": 12000,
    "Webcam": 4000,
    "USB Hub": 2000,
    "Printer Laser": 9000,
    "Printer Ink": 7000,
    "Printer MFP": 15000,
    "Printer Photo": 11000,
    "Power Bank": 3000,
    "Bagpack": 2500,
    "Screen Protector": 1000,
    "Cooling Pad": 1800,
    "Cable Set": 1200,
    "Server Rack": 150000,
    "Server Blade": 200000,
    "Storage NAS": 80000,
    "Network Switch": 25000,
    "Firewall": 45000,
}

channels = ["Интернет-магазин", "Офлайн-точка"]
client_types = ["Розница", "Опт"]
statuses = ["Доставлен", "В пути", "Отменён"]

months = {
    1: "Январь",
    2: "Февраль",
    3: "Март",
    4: "Апрель",
    5: "Май",
    6: "Июнь",
    7: "Июль",
    8: "Август",
    9: "Сентябрь",
    10: "Октябрь",
    11: "Ноябрь",
    12: "Декабрь",
}
# ---------- Функция генерации одной записи ----------
def generate_row(order_id):
    date = random.choice(date_list)
    # Русское название месяца (принудительно)
    month_ru = months[date.month]
    day = date.day

    manager = random.choice(names)
    city = random.choice(cities)
    region = cities_regions[city]
    category = random.choice(list(products_dict.keys()))
    product = random.choice(products_dict[category])

    client_type = random.choice(client_types)
    if client_type == "Опт":
        quantity = random.randint(5, 50)
        discount = 0.15
    else:
        quantity = random.randint(1, 10)
        discount = 0.0

    base_price = base_prices[product]
    price_noise = np.random.normal(1.0, 0.05)
    price_per_unit = base_price * price_noise * (1 - discount)
    price_per_unit = round(price_per_unit / 100) * 100
    revenue = quantity * price_per_unit

    channel = random.choice(channels)
    status = random.choice(statuses)

    return [
        order_id,
        date,
        day,
        month_ru,
        manager,
        city,
        region,
        category,
        product,
        client_type,
        quantity,
        price_per_unit,
        revenue,
        channel,
        status,
    ]


# ---------- Генерация 5000 строк ----------
num_rows = 5000
data = [generate_row(i) for i in range(1, num_rows + 1)]

columns = [
    "order_id",
    "date",
    "day",
    "month",
    "manager",
    "city",
    "region",
    "category",
    "product",
    "client_type",
    "quantity",
    "price_per_unit",
    "revenue",
    "channel",
    "status",
]

df = pd.DataFrame(data, columns=columns)

# ---------- Сохранение в Excel начиная с ячейки A2 ----------
if Path(file_path).exists():
    # Режим 'a' – добавляем/обновляем лист
    with pd.ExcelWriter(
        file_path, engine="openpyxl", mode="a", if_sheet_exists="replace"
    ) as writer:
        df.to_excel(writer, sheet_name=sheet_name, startrow=1, index=False)
else:
    # Файла нет – создаём с режимом 'w'
    with pd.ExcelWriter(file_path, engine="openpyxl", mode="w") as writer:
        df.to_excel(writer, sheet_name=sheet_name, startrow=1, index=False)

print(f"Всего записей: {len(df)}")
print("Первые 5 строк датасета:")
print(df.head())


Всего записей: 5000
Первые 5 строк датасета:
   order_id       date  day    month   manager       city       region  \
0         1 2025-11-13   13   Ноябрь   Дмитрий     Самара  Приволжский   
1         2 2025-12-24   24  Декабрь  Светлана       Омск    Сибирский   
2         3 2025-07-13   13     Июль  Светлана  Краснодар        Южный   
3         4 2025-08-26   26   Август     Игорь        Уфа  Приволжский   
4         5 2025-05-09    9      Май     Мария    Воронеж  Центральный   

   category         product client_type  quantity  price_per_unit  revenue  \
0  Мониторы      Monitor 24     Розница        10           16800   168000   
1  Ноутбуки  Notebook Ultra         Опт        33           96100  3171300   
2  Принтеры     Printer Ink     Розница         6            7300    43800   
3  Принтеры     Printer MFP     Розница         8           15100   120800   
4  Ноутбуки    Notebook Air     Розница         3           63400   190200   

            channel     status  
0      О

In [30]:
import pandas as pd
from pathlib import Path

# TODO:
# 1. Загрузите лист sales_data в DataFrame df
# 2. Создайте столбец revenue = quantity * price
# 3. Выведите размер таблицы
# 4. Посчитайте количество заказов
# 5. Посчитайте общую, среднюю, максимальную и минимальную выручку
# 6. Покажите первые строки таблицы
print(file_path)
df = pd.read_excel(file_path, sheet_name="sales_data", header=1)

print("Размер таблицы:", df.shape)
print("Количество заказов:", df["order_id"].count())
print("Общая выручка:", round(df["revenue"].sum()))
print("Средняя выручка заказа:", round(df["revenue"].mean(), 2))
print("Максимальная выручка заказа:", round(df["revenue"].max()))
print("Минимальная выручка заказа:", round(df["revenue"].min()))

df.head()


..\..\xls\lesson_09_groupby_pivot.xlsx
Размер таблицы: (5000, 15)
Количество заказов: 5000
Общая выручка: 2510063300
Средняя выручка заказа: 502012.66
Максимальная выручка заказа: 9074800
Минимальная выручка заказа: 1000


,order_id,date,day,month,manager,city,region,category,product,client_type,quantity,price_per_unit,revenue,channel,status
0,1,2025-11-13,13,Ноябрь,Дмитрий,Самара,Приволжский,Мониторы,Monitor 24,Розница,10,16800,168000,Офлайн-точка,В пути
1,2,2025-12-24,24,Декабрь,Светлана,Омск,Сибирский,Ноутбуки,Notebook Ultra,Опт,33,96100,3171300,Офлайн-точка,Доставлен
2,3,2025-07-13,13,Июль,Светлана,Краснодар,Южный,Принтеры,Printer Ink,Розница,6,7300,43800,Офлайн-точка,Отменён
3,4,2025-08-26,26,Август,Игорь,Уфа,Приволжский,Принтеры,Printer MFP,Розница,8,15100,120800,Интернет-магазин,Отменён
4,5,2025-05-09,9,Май,Мария,Воронеж,Центральный,Ноутбуки,Notebook Air,Розница,3,63400,190200,Интернет-магазин,Доставлен


## Практическая ячейка 2. Группировка по менеджеру

### Что делаем
Группируем данные по столбцу `manager` и считаем для каждого менеджера:

- общую выручку;
- количество заказов;
- общее количество проданных единиц;
- среднюю выручку заказа.

### Функции, методы и синтаксис
- `df.groupby("column")` — группирует строки по значениям столбца.
- `.agg(...)` — задаёт, какие агрегаты нужно посчитать.
- `as_index=False` — сохраняет столбец группы как обычный столбец, а не как индекс.
- `reset_index()` — возвращает группировку в обычную таблицу, если группа стала индексом.


In [31]:
# TODO:
# 1. Сгруппируйте df по столбцу manager
# 2. Посчитайте total_revenue, orders, total_quantity, avg_revenue
# 3. Сохраните результат в manager_summary
# 4. Округлите avg_revenue до 2 знаков
# 5. Покажите результат
manager_summary = df.groupby("manager", as_index=False).agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    total_quantity=("quantity", "sum"),
    avg_revenue=("revenue", "mean"),
)
manager_summary["avg_revenue"] = manager_summary["avg_revenue"].round(2)

manager_summary


,manager,total_revenue,orders,total_quantity,avg_revenue
0,Алексей,192949900,370,5972,521486.22
1,Анна,198233100,322,5530,615630.75
2,Борис,168579000,352,5784,478917.61
3,Владимир,211956100,363,6035,583901.10
4,Дмитрий,139977400,322,5167,434712.42
5,Елена,186721400,324,5658,576300.62
6,Игорь,153184100,324,4988,472790.43
7,Ксения,165682400,321,5870,516144.55
8,Мария,173147800,351,6147,493298.58
9,Наталья,179529100,333,5857,539126.43


## Практическая ячейка 3. Группировка по двум признакам: `month` и `category`

### Что делаем
Считаем агрегаты сразу по двум столбцам:

- месяц;
- категория товара.

Так мы увидим, какая категория и в каком месяце дала больше выручки.

### Функции, методы и синтаксис
- `df.groupby(["col1", "col2"])` — группировка сразу по двум признакам.
- `pd.Categorical(...)` — позволяет задать правильный порядок месяцев.
- `.sort_values([...])` — сортирует итоговую таблицу по выбранным столбцам.


In [32]:
# TODO:
# 1. Задайте правильный порядок месяцев в списке month_order
# 2. Преобразуйте df["month"] в категориальный столбец с этим порядком
# 3. Сгруппируйте данные по month и category
# 4. Посчитайте total_revenue, total_quantity и orders
# 5. Отсортируйте результат по month и total_revenue по убыванию внутри месяца
# 6. Сохраните результат в month_category_summary

month_category_summary = pd.Categorical(df["month"], categories=months, ordered=True)

month_category_summary = (
    df.groupby(["month", "category"], as_index=False, observed=True)
        .agg(
            total_revenue=("revenue", "sum"),
            total_quantity=("quantity", "sum"),
            orders=("order_id", "count")
        )
        .sort_values(["month", "total_revenue"], ascending=[False, False])
)

month_category_summary


C:\Temp\ipykernel_13600\1102493483.py:9: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  month_category_summary = pd.Categorical(df["month"], categories=months, ordered=True)


,month,category,total_revenue,total_quantity,orders
71,Январь,Серверы,102323000,1038,68
68,Январь,Ноутбуки,55249900,882,57
67,Январь,Мониторы,25511400,1503,78
70,Январь,Принтеры,11017000,1218,78
69,Январь,Периферия,6645800,1300,66
...,...,...,...,...,...
2,Август,Ноутбуки,70242600,1008,69
1,Август,Мониторы,21565500,1356,71
4,Август,Принтеры,10533100,1101,74
3,Август,Периферия,5294800,1091,68


## Практическая ячейка 4. Сортируем итоговые таблицы и ищем лидера

### Что делаем
Берём результат группировки по менеджерам и сортируем его по `total_revenue` по убыванию.  
После этого легко определить лидера продаж.

### Функции, методы и синтаксис
- `sort_values(by="column", ascending=False)` — сортировка по убыванию.
- `iloc[0]` — первая строка после сортировки.
- `["column"]` — получение конкретного значения из строки или таблицы.


In [42]:
# TODO:
# 1. Отсортируйте manager_summary по total_revenue по убыванию
# 2. Сохраните результат в manager_sorted
# 3. Найдите имя лидера в переменной top_manager
# 4. Найдите его выручку в переменной top_manager_revenue
# 5. Выведите результаты
# 6. Покажите таблицу manager_sorted

manager_sorted = manager_summary.sort_values(by="total_revenue", ascending=False)

top_manager = manager_sorted.iloc[0]["manager"]
top_manager_revenue = manager_sorted.iloc[0]["total_revenue"]

print("Лидер по выручке:", top_manager)
print("Выручка лидера:", top_manager_revenue)

manager_sorted


Лидер по выручке: Владимир
Выручка лидера: 211956100


,manager,total_revenue,orders,total_quantity,avg_revenue
3,Владимир,211956100,363,6035,583901.10
1,Анна,198233100,322,5530,615630.75
0,Алексей,192949900,370,5972,521486.22
5,Елена,186721400,324,5658,576300.62
9,Наталья,179529100,333,5857,539126.43
8,Мария,173147800,351,6147,493298.58
2,Борис,168579000,352,5784,478917.61
7,Ксения,165682400,321,5870,516144.55
14,Татьяна,157598100,328,5066,480482.01
11,Павел,153711500,299,5389,514085.28


## Практическая ячейка 5. Строим мини-сводную таблицу через `pivot_table`

### Что делаем
Создаём мини-сводную таблицу, где:

- строки — это менеджеры;
- столбцы — категории;
- значения — сумма `revenue`.

После этого делаем короткий вывод:
- какая категория даёт больше всего выручки;
- какой менеджер сильнее всего в категории `Ноутбуки`.

### Функции, методы и синтаксис
- `pd.pivot_table(...)` — создаёт сводную таблицу.
- `values="revenue"` — что агрегируем.
- `index="manager"` — строки сводной таблицы.
- `columns="category"` — столбцы сводной таблицы.
- `aggfunc="sum"` — суммируем выручку.
- `fill_value=0` — заменяем пустые ячейки на ноль.
- `sum(axis=0)` — сумма по столбцам.
- `idxmax()` — индекс или название столбца/строки с максимальным значением.


In [48]:
# TODO:
# 1. Постройте pivot_table по выручке:
#    строки — manager, столбцы — category, значения — revenue, агрегат — sum
# 2. Замените пропуски на 0 через fill_value=0
# 3. Сохраните результат в pivot
# 4. Оставьте столбцы в порядке: Ноутбуки, Мониторы, Периферия, Принтеры
# 5. Найдите лучшую категорию по выручке в переменной best_category
# 6. Найдите менеджера-лидера по категории "Ноутбуки" в переменной best_notebook_manager
# 7. Выведите результаты и покажите сводную таблицу

pivot = pd.pivot_table(
    df,
    values="revenue",
    index="manager",
    columns="category",
    aggfunc="sum",
    fill_value=0,
)

# pivot = pivot[["Ноутбуки", "Мониторы", "Периферия", "Принтеры"]]

best_category = pivot.sum(axis=0).idxmax()
best_notebook_manager = pivot[best_category].idxmax()

print("Лучшая категория по выручке:", best_category)
print(f"Сильнейший менеджер в категории '{best_category}':", best_notebook_manager)

pivot


Лучшая категория по выручке: Серверы
Сильнейший менеджер в категории 'Серверы': Владимир


category,Аксессуары,Мониторы,Ноутбуки,Периферия,Принтеры,Серверы
manager,,,,,,
Алексей,1506200,12685700,70746000,5225700,9174700,93611600
Анна,1646000,13707100,70733800,3259400,8710900,100175900
Борис,1597800,16282800,51111900,3755600,9955900,85875000
Владимир,1732400,19165200,58760900,2665000,10007300,119625300
Дмитрий,1564900,15591100,50768400,3429700,6359500,62263800
Елена,1592600,22097200,60522900,3238000,6903400,92367300
Игорь,1270600,12834600,52369900,3514500,9408600,73785900
Ксения,1703800,17133100,40892400,7733500,6852100,91367500
Мария,2036300,17592900,60994800,3567300,9608300,79348200


## Ячейка 6. Тест и самопроверка

### Что делаем
Проверяем, что вы умеете:

- загружать Excel-лист;
- считать агрегаты;
- делать `groupby` по одному и двум признакам;
- сортировать итоговые таблицы;
- строить мини-сводную таблицу.

### Функции, методы и синтаксис
- `assert условие` — если условие ложное, Python покажет ошибку.
- `groupby()`, `agg()`, `sort_values()`, `pivot_table()` — основные инструменты этого занятия.


In [54]:
import pandas as pd

test_df = pd.read_excel(file_path, sheet_name="sales_data", header=1)
test_df["revenue"] = test_df["quantity"] * test_df["price_per_unit"]

assert test_df.shape == (5000, 15)
assert int(test_df["order_id"].count()) == 5000
assert int(test_df["revenue"].sum()) == 2612584200
assert int(test_df["revenue"].max()) == 8451000
assert int(test_df["revenue"].min()) == 900

test_manager_summary = (
    test_df.groupby("manager", as_index=False)
           .agg(
               total_revenue=("revenue", "sum"),
               orders=("order_id", "count"),
               total_quantity=("quantity", "sum")
           )
)

anna_revenue = int(test_manager_summary.loc[test_manager_summary["manager"] == "Анна", "total_revenue"].iloc[0])
boris_orders = int(test_manager_summary.loc[test_manager_summary["manager"] == "Борис", "orders"].iloc[0])

assert anna_revenue == 185913300
assert boris_orders == 308

month_order = ["Январь", "Февраль", "Март", "Апрель", "Май", "Июнь", "Июль", "Август", "Сентябрь", "Октябрь", "Ноябрь", "Декабрь"]
test_df["month"] = pd.Categorical(test_df["month"], categories=month_order, ordered=True)

test_month_category = (
    test_df.groupby(["month", "category"], as_index=False, observed=True)
           .agg(total_revenue=("revenue", "sum"))
)

jan_notebooks = int(
    test_month_category[
        (test_month_category["month"] == "Январь") &
        (test_month_category["category"] == "Ноутбуки")
    ]["total_revenue"].iloc[0]
)

assert jan_notebooks == 72690500

test_sorted = test_manager_summary.sort_values(by="total_revenue", ascending=False)
assert int(test_sorted.iloc[0]["total_revenue"]) == 203580400

test_pivot = pd.pivot_table(
    test_df,
    values="revenue",
    index="manager",
    columns="category",
    aggfunc="sum",
    fill_value=0
)

assert int(test_pivot.loc["Борис", "Ноутбуки"]) == 51621200
assert int(test_pivot.sum(axis=0).loc["Ноутбуки"]) == 960294200

print("Тест пройден успешно.")


Тест пройден успешно.
